# 01 - Correccion y filtrado

Corrige linea base, aplica filtro Butterworth de fase cero, integra a velocidad/desplazamiento y guarda metricas, espectro y figuras.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import signalprocessor
print(f"ROOT={ROOT}")
print(f"signalprocessor={Path(signalprocessor.__file__).resolve()}")

In [ ]:
from signalprocessor.io import read_motion_csv, write_json, write_motion_csv, write_spectrum_csv
from signalprocessor.plotting import save_motion_plot, save_spectrum_plot
from signalprocessor.processing import ProcessConfig, process_motion

OUT = ROOT / "examples" / "ouputs" / "case_01_correccion_filtrado"
OUT.mkdir(parents=True, exist_ok=True)

motion = read_motion_csv(ROOT / "examples" / "data" / "motion" / "LIMAEW.csv", unit="g", name="LIMAEW")
config = ProcessConfig(
    baseline_method="polynomial",
    baseline_order=1,
    enforce_zero_end=True,
    highpass_hz=0.08,
    lowpass_hz=18.0,
    filter_order=4,
    taper_fraction=0.02,
    pad_seconds=5.0,
    spectrum_min_period=0.02,
    spectrum_max_period=5.0,
    spectrum_points=100,
)
ouput = process_motion(motion, config)

write_motion_csv(OUT / "LIMAEW_corregido_filtrado.csv", ouput.filtered.motion, unit="g")
write_json(OUT / "LIMAEW_metricas.json", ouput.metrics | {"baseline": ouput.baseline.info, "filter": ouput.filtered.info})
write_spectrum_csv(OUT / "LIMAEW_espectro.csv", ouput.spectrum["period_s"], ouput.spectrum["sa_g"])
save_motion_plot(OUT / "LIMAEW_corregido_filtrado.png", ouput.filtered.motion, ouput.velocity_m_s, ouput.displacement_m, title="LIMAEW corregido y filtrado")
save_spectrum_plot(OUT / "LIMAEW_espectro.png", ouput.spectrum["period_s"], {"Sa 5%": ouput.spectrum["sa_g"]})

ouput.metrics